# HyperReason v2 — quickstart (no local install)

Runs the **real** AE-MCTS engine against the Z.AI GLM gateway, server-side in this Colab runtime.
Your API key stays in the runtime — it never crosses a static web page.

1. Put your Z.AI key in the Colab secret `ZAI_KEY` (or `ANTHROPIC_API_KEY`).
2. Run all.

In [ ]:
!pip install -q "git+https://github.com/rudra496/hyper-reason.git@v2/honest-rebuild" requests
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ZAI_KEY') or userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass
os.environ.setdefault('ANTHROPIC_BASE_URL', 'https://api.z.ai/api/anthropic')
assert os.environ.get('ANTHROPIC_API_KEY'), 'set the ZAI_KEY / ANTHROPIC_API_KEY Colab secret'

In [ ]:
from hyper_reason import wrap_model, GLMBackend, SearchPresets

model = wrap_model(GLMBackend(model='glm-4.6'), config=SearchPresets.balanced())
res = model.reason('Janet has 3 boxes of 12 apples. She gives away 5 and eats 2. How many remain?')
print('boxed answer :', res['boxed_answer'])
print('confidence    :', res['confidence'])
print('model calls   :', res['metrics']['model_calls'])
print('entropy src   :', res['metrics']['config']['entropy_source'])
print('flashkv       :', res['metrics']['flashkv'])

In [ ]:
# Reproduce the headline eval (small N here; raise --n for the full run)
!python eval/gsm8k_mini.py --n 10 --backend glm --model glm-4.6 --sims 6 --k 2 --depth 3
run = sorted(__import__('glob').glob('eval/runs/*.jsonl'))[-1]
!python eval/aggregate.py {run}